# 02 - Upgraded Earth Engine extraction (v2): more stations, more regions, daily drivers

Run this in **Google Colab** with your Earth Engine project. Upload the `gee/` folder, or clone your repo.

It was **not executed** where it was written, because there was no Earth Engine access there. The collection IDs
and band names were checked against the public catalogue. The assembly step (`assemble_v2.py`) was tested on
synthetic exports with the same column layout. Start with the 16 paper stations and 1 year, check the CSVs,
then scale up.

**What it fixes or adds versus v1**
1. Station coordinates now come from NRCS/ISMN metadata. In v1, all 16 stations were 2-360 km off.
2. Features are averaged over a 50 m buffer instead of a single 10 m pixel.
3. S1 is read in linear power. Ratio, RVI and SPAN are computed correctly. Incidence angle and orbit are kept for change detection.
4. S2 uses the SCL pixel cloud mask. MODIS uses BRDF-corrected NBAR with QA, plus LAI and day/night LST.
5. Daily drivers are added: CHIRPS, ERA5-Land, and SMAP L4 / L3-enhanced (the passive-microwave prior in your project plan).
6. Static descriptors are added: SoilGrids, SRTM, MERIT-Hydro TWI and ESA WorldCover.
7. Collection-level `reduceRegions` exports scale to hundreds of stations.

In [ ]:
!pip -q install earthengine-api ismn pandas pyarrow
import sys; sys.path.insert(0, 'gee')   # folder with gee_extract_v2.py, insitu_v2.py, assemble_v2.py
import pandas as pd
import gee_extract_v2 as G
G.init('YOUR-GEE-PROJECT')

## 1. Stations
* `stations_paper16_corrected.csv`: your 16 stations with the NRCS coordinates.
* `stations_scan_conus_candidates.csv`: all 192 CONUS SCAN sites, with a climate region for each.
* USCRN: about 115 stations. They have 5 cm soil moisture plus precipitation and temperature measured at the same site.
* ISMN: download networks from ismn.earth. Examples are REMEDHUS (ES), SMOSMANIA (FR), TERENO (DE), OZNET (AU), MAQU / CTP_SMTMN (CN) and SOILSCAPE (US). Then use `insitu_v2.ismn_daily()`.

In [ ]:
st16  = G.load_stations('gee/stations_paper16_corrected.csv')
scan  = pd.read_csv('gee/stations_scan_conus_candidates.csv').rename(columns={'scan_id':'sid'})
print(scan.region.value_counts())
# e.g. all SCAN stations that started before 2019:
scan_sel = scan[pd.to_datetime(scan.start, format='%Y-%B', errors='coerce') < '2019-01-01']
stations = pd.concat([st16[['sid','lat','lon']], scan_sel[['sid','lat','lon']]]).drop_duplicates('sid')
len(stations)

## 2. Start the exports (one Drive task per product). Run one call per year for many stations.

In [ ]:
tasks = {}
for y in range(2019, 2024):
    tasks[y] = G.export_all(stations, f'{y}-01-01', f'{y+1}-01-01', folder='sm_v2', prefix=f'y{y}_',
                            products=['s1','s2','nbar','lai','lst','chirps','era5l','smap_l4','smap_l3e'])
G.export_all(stations, '2019-01-01', '2019-01-02', folder='sm_v2', products=['static'])

## 3. In-situ data
SCAN uses the Report Generator CSV service, USCRN uses the daily01 files, and ISMN uses an archive folder.

In [ ]:
import insitu_v2 as I
frames = []
for r in pd.read_csv('gee/stations_scan_conus_candidates.csv').itertuples():
    try:
        frames.append(I.scan_daily(r.station_triplet, '2019-01-01', '2023-12-31'))
    except Exception as e:
        print(r.station_triplet, e)
insitu = pd.concat(frames)
insitu.to_parquet('insitu_daily.parquet')

## 4. Assemble the model tables (after the Drive exports finish, mount Drive)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!python gee/assemble_v2.py --exports /content/drive/MyDrive/sm_v2 --insitu insitu_daily.parquet \
        --stations gee/stations_scan_conus_candidates.csv --start 2019-01-01 --end 2023-12-31 --out data/processed_v2